# Parse XML Strings

I want to see if I can parse the XML strings that contain transcriptions of apparatus entries, with the goal of validating the claims. The idea is to parse a string like this:

```xml
<ab><hi rend="italics">subdentur</hi> νφρηθrs, <hi rend="italics">subiguntur</hi> P, <hi rend="italics">claudentur</hi> Ulit.</ab>
```
… so that it can be used to check a JSON representation.

I'm pretty sure that I can use the fact that entries are delineated by commas to break up the transcription into parts that map onto the claims. And I'm pretty sure that I can do some string parsing to get the individual sigla for the sources. We shall see.

In [108]:
import json
import re
from lxml import etree
from itertools import zip_longest

In [102]:
def remove_parentheticals(transcription):
    return re.sub(r"\([^()]*\)", "", transcription)


def parse_xml_string(xml_string):
    """
    Parses an XML string and extracts readings and their corresponding evidence.
    
    Args:
        xml_string (str): The XML string to parse.

    Returns:
        list: A list of tuples, where each tuple contains a reading and its corresponding evidence.
    """
    # Strip the escaping characters from the XML string
    stripped_xml_string = xml_string.replace("\\", "")
    cleaned = remove_parentheticals(stripped_xml_string)
    # Parse the cleaned XML string into an XML tree
    try:
        root = etree.fromstring(cleaned.encode("utf-8"))
    except etree.XMLSyntaxError as e:
        print(f"XML parsing error: {e}")
        print(f"Original transcription:\n{xml_string}")
        print(f"Cleaned transcription:\n{cleaned}")
        raise

    # Extract readings and their corresponding evidence
    readings = []

    for hi in root.xpath("./hi[@rend='italic']"):
        # Extract the reading text from the <hi rend="italic"> element
        reading = extract_reading(hi)
        # Extract the evidence text, which is the tail of the <hi> element, and clean it up
        evidence = (hi.tail or "").strip().rstrip(",")
        readings.append((reading, evidence))

    return readings


def handle_labels_in_evidence(evidence):
    label = evidence.get("label")
    hand = evidence.get("hand")

    return label + str(hand) if hand is not None else label


def extract_reading(hi):
    reading = "".join(hi.itertext())

    if reading in {"O", "C"}:
        if hi.tail and hi.tail.startswith("."):
            reading += "."

    return reading


def normalize_evidence(text):
    # Remove parenthetical expressions, including nested parentheses
    result = []
    depth = 0

    for char in text:
        if char == "(":
            depth += 1
        elif char == ")":
            if depth > 0:
                depth -= 1
        elif depth == 0:
            result.append(char)

    text = "".join(result)

    # Remove terminal punctuation, preserving periods in abbreviations
    text = text.strip().rstrip(",;")

    return text.strip()


def tokenize_evidence(text, labels):
    text = normalize_evidence(text)

    labels = sorted(labels, key=len, reverse=True)
    pattern = re.compile("|".join(re.escape(label) for label in labels))

    tokens = []
    position = 0

    while position < len(text):
        char = text[position]

        # Ignore whitespace and punctuation separators
        if char.isspace() or char in ",;]":
            position += 1
            continue

        # Ignore "om." and "add." as evidence tokens
        if text[position : position + 3] in {"om.", "add."}:
            position += 3
            continue
        
        match = pattern.match(text, position)

        if match:
            label = match.group()
            position = match.end()

            # Check for a hand number immediately following the label
            hand = ""

            while position < len(text) and text[position].isdigit():
                hand += text[position]
                position += 1

            tokens.append(label + hand)
            continue

        elif char == ".":
            # Ignore a period not recognized as part of a label
            position += 1

        else:
            raise ValueError(
                f"Unrecognized evidence at position {position}: " f"{text[position:]!r}"
            )

    return tokens

In [61]:
# Open the edition profile JSON file and load its contents
with open(
    "../converted-json/phi0830.phi001.new_edition_profile2.json", "r", encoding="utf-8"
) as f:
    edition_profile = json.load(f)

edition_profile_items = ["sigla", "groups", "secondary_sources", "people"]
edition_profile_labels = set()
for item in edition_profile_items:
    if item in edition_profile:
        for entry in edition_profile[item]:
            label = entry.get("label")
            if label:
                edition_profile_labels.add(label)

In [42]:
edition_profile_labels

{'A',
 'Argol.',
 'Ascensius',
 'Aurel.',
 'Baehr.',
 'Baehr. 1870',
 'Baehr. 1872',
 'Barth',
 'Barth 1624',
 'Barth 1650',
 'Barthius',
 'Bartholinus',
 'Beck',
 'Bergk',
 'Brantsma',
 'Brodaeus',
 'Buecheler 1860',
 'Buecheler 1871',
 'Buecheler 1907',
 'Burm.',
 'Burm. 1759',
 'Bursian',
 'C. Schenkl',
 'Carrio',
 'Casaub.',
 'Chytil',
 'Dempster',
 'D’Orville',
 'Ellis',
 'Erasmus',
 'Exc. Bon.',
 'Exc. Par.',
 'Exc. Par. Alter',
 'Exc. Par. Prior',
 'Fabricius',
 'Forbiger',
 'Friesemann',
 'Fritzsche',
 'Fruterius',
 'G',
 'G. Hermann',
 'Gebauer',
 'Gebhard',
 'Giarratano',
 'Glaeser',
 'Gronov.',
 'Gronov. 1637',
 'Gryph.',
 'Gudius',
 'Guid.',
 'H',
 'Hartel',
 'Haupt 1854',
 'Haupt 1874',
 'Heins.',
 'Heins. ad Claud.',
 'Heins. in Her.',
 'Heraldus',
 'Hoeufft',
 'Housman',
 'Housman 1902',
 'Housman 1903',
 'Iustus',
 'Jacoby',
 'Kannegieter',
 'Keene',
 'Kempfer',
 'Lachmann',
 'Leo',
 'Lipsius',
 'Lucas',
 'Maehly',
 'Magnus',
 'Martell.',
 'Modius',
 'Mueller',
 'N',
 '

In [39]:
original_transcription = "<ab><hi rend=\"italic\">subdentur</hi> νφρηθrs (lorem ipsum), <hi rend=\"italic\">subiguntur</hi> P, <hi rend=\"italic\">claudentur</hi> Ulit.</ab>"

In [40]:
parsed_xml = parse_xml_string(original_transcription)

In [80]:
type(parsed_xml)

list

In [44]:
for reading, evidence in parsed_xml:
    try:
        tokens = tokenize_evidence(evidence, edition_profile_labels)
        print(f"Reading: {reading}, Evidence Tokens: {tokens}")
    except ValueError as e:
        print(f"Error tokenizing evidence for reading '{reading}': {e}")

Reading: subdentur, Evidence Tokens: ['ν', 'φ', 'ρ', 'η', 'θ', 'r', 's']
Reading: subiguntur, Evidence Tokens: ['P']
Reading: claudentur, Evidence Tokens: ['Ulit.']


In [118]:
# Read in the apparatus data from the JSON file
with open("../converted-json/merged-apparatus-entries-and-claims-valid-cts-urns.json", "r", encoding="utf-8") as f:
    apparatus_data = json.load(f)

In [77]:
entries = []
for entry in apparatus_data:
    json_data = {}
    entry_id = entry.get("id")
    transcription = entry.get("transcription")
    if transcription:
        json_data["original_transcription"] = transcription
        parsed_xml = parse_xml_string(transcription)
        transcription_parts = []
        for reading, evidence in parsed_xml:
            transcription_dict = {}
            try:
                tokens = tokenize_evidence(evidence, edition_profile_labels)
                transcription_dict["reading"] = reading
                transcription_dict["evidence"] = tokens
            except ValueError as e:
                print(f"Error in entry '{entry_id}', tokenizing evidence for reading '{reading}': {e}")
            transcription_parts.append(transcription_dict)
        json_data["transcription"] = transcription_parts
    claims = []
    for claim in entry.get("claims", []):
        claim_id = claim.get("id")
        claim_dict = {}
        claim_reading = claim.get("reading")
        claim_labels = []
        for evidence in claim.get("supporting_evidence", []):
            evidence_label = handle_labels_in_evidence(evidence)
            if evidence_label:
                claim_labels.append(evidence_label)
        if claim_labels is not None:
            try:
                tokens = tokenize_evidence(" ".join(claim_labels), edition_profile_labels)
                claim_dict["reading"] = claim_reading
                claim_dict["evidence"] = tokens
            except ValueError as e:
                print(f"Error in entry '{entry_id}', claim '{claim_id}', tokenizing claim evidence: {e}")
        claims.append(claim_dict)
    json_data["claims"] = claims
    entries.append(json_data)

Error in entry 'urn:uuid:90c8a150-48bf-43e3-b9b2-8ae8d02b2d6f', tokenizing evidence for reading 'C. ': Unrecognized evidence at position 7: 'que, edd. fere omnes ante Glaeser.'
Error in entry 'urn:uuid:2a736434-b584-430d-b642-42d28de68370', tokenizing evidence for reading 'ulmis': Unrecognized evidence at position 9: 'apud Titium.'
Error in entry 'urn:uuid:2c98e1e5-f190-4e8f-8c63-5c0b46988aed', tokenizing evidence for reading 'O.': Unrecognized evidence at position 1: 'm. N .'
Error in entry 'urn:uuid:9d0a2c08-9e17-4dc5-b987-363db1accb16', tokenizing evidence for reading 'antra': Unrecognized evidence at position 0: '-9'
Error in entry 'urn:uuid:9d0a2c08-9e17-4dc5-b987-363db1accb16', tokenizing evidence for reading 'ista': Unrecognized evidence at position 0: '-9'
Error in entry 'urn:uuid:abfa04d2-c723-4732-8274-e7efa574ff08', tokenizing evidence for reading 'ubi': Unrecognized evidence at position 0: '] G .'
Error in entry 'urn:uuid:059560c1-2d15-4988-8320-0b81080d38a8', tokenizing ev

In [78]:
entries

[{'original_transcription': '<ab>I 1 <hi rend="italic">C</hi>. G P A φ Ulit. Wernsd. Glaeser sqq., <hi rend="italic">om.</hi> Nπχ p, <hi rend="italic">O</hi>. εβγμρ edd. ante Glaeser.</ab>',
  'transcription': [{'reading': 'C.',
    'evidence': ['G', 'P', 'A', 'φ', 'Ulit.', 'Wernsd.', 'Glaeser', 'sqq.']},
   {'reading': 'om.', 'evidence': ['N', 'π', 'χ', 'p']},
   {'reading': 'O.',
    'evidence': ['ε', 'β', 'γ', 'μ', 'ρ', 'edd. ante', 'Glaeser']}],
  'claims': [{'reading': 'C.',
    'evidence': ['G', 'P', 'A', 'φ', 'Ulit.', 'Wernsd.']},
   {'reading': None, 'evidence': ['N', 'π', 'χ', 'p']},
   {'reading': 'O.', 'evidence': ['ε', 'β', 'γ', 'μ', 'ρ']}]},
 {'original_transcription': '<ab><hi rend="italic">nundum</hi> G (corr. m<hi rend="superscript">1</hi>) P.</ab>',
  'transcription': [{'reading': 'nundum', 'evidence': ['G', 'P']}],
  'claims': [{'reading': 'nundum', 'evidence': ['G', 'P']}]},
 {'original_transcription': '<ab><hi rend="italic">declinis</hi> N Heins. Schenkl, <hi rend="

In [141]:
def parse_transcription(xml_string):
    parsed_xml = parse_xml_string(xml_string)
    transcription_parts = []
    for reading, evidence in parsed_xml:
        transcription_dict = {
            "reading": reading,
            "evidence": None,
        }
        try:
            tokens = tokenize_evidence(evidence, edition_profile_labels)
            transcription_dict["reading"] = reading
            transcription_dict["evidence"] = tokens
        except ValueError as e:
            print(f"Error tokenizing evidence for reading '{reading}': {e}")
        transcription_parts.append(transcription_dict)
    return transcription_parts


def parse_claims(claims):
    parsed_claims = []
    for claim in claims:
        claim_dict = {}
        claim_reading = claim.get("reading")
        claim_labels = []
        for evidence in claim.get("supporting_evidence", []):
            evidence_label = handle_labels_in_evidence(evidence)
            if evidence_label:
                claim_labels.append(evidence_label)
        if claim_labels is not None:
            try:
                tokens = tokenize_evidence(" ".join(claim_labels), edition_profile_labels)
                claim_dict["reading"] = claim_reading
                claim_dict["evidence"] = tokens
            except ValueError as e:
                print(f"Error tokenizing claim evidence for reading '{claim_reading}': {e}")
        parsed_claims.append(claim_dict)
    return parsed_claims


def build_validation_result(entry, transcription_readings, claim_readings, log_file):

    if len(transcription_readings) != len(claim_readings):
        message = (
            f"Warning: Entry {entry['id']}: "
            f"Number of transcription readings ({len(transcription_readings)}) "
            f"does not match number of claims ({len(claim_readings)})."
        )
        log_file.write(message + "\n")

        not_equal = {
            "entry_id": entry["id"],
            "transcription": transcription_readings,
            "claims": claim_readings,
            "status": "count_mismatch",
        }
        log_file.write(json.dumps(not_equal, indent=4) + "\n")
        return not_equal
    else:
        print(
            f"Number of transcription readings ({len(transcription_readings)}) "
            f"matches number of claims ({len(claim_readings)})."
        )
        comparisons = []
        for transcription, claim in zip_longest(
            transcription_readings, claim_readings, fillvalue=None
        ):
            if transcription is None or claim is None:
                comparisons.append(
                    {
                        "reading": (
                            transcription["reading"] if transcription else claim["reading"]
                        ),
                        "transcription_evidence": (
                            transcription["evidence"] if transcription else None
                        ),
                        "claim_evidence": (claim["evidence"] if claim else None),
                        "reading_match": False,
                        "evidence_match": False,
                        "status": "missing_record",
                    }
                )
                continue

            transcription_reading = transcription.get("reading")
            claim_reading = claim.get("reading")

            transcription_evidence = transcription.get("evidence")
            claim_evidence = claim.get("evidence")

            comparisons.append(
                {
                    "reading": transcription_reading,
                    "claim_reading": claim_reading,
                    "transcription_evidence": transcription_evidence,
                    "claim_evidence": claim_evidence,
                    "reading_match": (
                        transcription_reading is not None
                        and claim_reading is not None
                        and transcription_reading == claim_reading
                    ),
                    "evidence_match": (
                        transcription_evidence is not None
                        and claim_evidence is not None
                        and transcription_evidence == claim_evidence
                    ),
                    "status": (
                        "missing_data"
                        if transcription_reading is None or claim_reading is None
                        else (
                            "match"
                            if (
                                transcription_reading == claim_reading
                                and transcription_evidence is not None
                                and claim_evidence is not None
                                and transcription_evidence == claim_evidence
                            )
                            else "mismatch"
                        )
                    ),
                    "reviewer": "SJH",
                    "reviewer_notes": None,
                    "review_status": (
                        "Passed"
                        if (
                            transcription_reading is not None
                            and claim_reading is not None
                            and transcription_reading == claim_reading
                            and transcription_evidence is not None
                            and claim_evidence is not None
                            and transcription_evidence == claim_evidence
                        )
                        else "Failed"
                    ),
                }
            )

        return {
            "entry_id": entry["id"],
            "text_locus": entry["text_locus"],
            "comparisons": comparisons,
        }

In [142]:
log_file_path = "../converted-json/validation_results2.md"
with open(log_file_path, "w", encoding="utf-8") as log_file:
    entries = []
    for entry in apparatus_data:
        print(f"Processing entry ID: {entry.get('id')}")
        transcription = parse_transcription(entry.get("transcription")) if entry.get("transcription") else []
        claims = parse_claims(entry.get("claims", []))
        result = build_validation_result(entry, transcription, claims, log_file)
        entries.append(result)

Processing entry ID: urn:uuid:f2e93fa5-f815-4ff4-92ca-adaa365f8f20
Number of transcription readings (3) matches number of claims (3).
Processing entry ID: urn:uuid:37b666e6-9980-4776-81d9-d056936f49c5
Number of transcription readings (1) matches number of claims (1).
Processing entry ID: urn:uuid:fc07310e-e32e-42f5-b6b8-7915871f71bc
Number of transcription readings (4) matches number of claims (4).
Processing entry ID: urn:uuid:c8f6d0a9-7c57-4a5e-9def-3ca62cf2e188
Number of transcription readings (1) matches number of claims (1).
Processing entry ID: urn:uuid:7757d8c7-4980-42be-bf21-28ccf1a2659f
Number of transcription readings (1) matches number of claims (1).
Processing entry ID: urn:uuid:022529cf-6b85-48c8-8dee-6b53245e3c44
Number of transcription readings (1) matches number of claims (1).
Processing entry ID: urn:uuid:4817a744-94b3-4f6c-a2ec-d57ca52daef9
Number of transcription readings (1) matches number of claims (1).
Processing entry ID: urn:uuid:90c8a150-48bf-43e3-b9b2-8ae8d02b

In [143]:
entries

[{'entry_id': 'urn:uuid:f2e93fa5-f815-4ff4-92ca-adaa365f8f20',
  'text_locus': ['urn:cts:latinLit:phi0830.phi001.dll:1.1@C.'],
  'comparisons': [{'reading': 'C.',
    'claim_reading': 'C.',
    'transcription_evidence': ['G',
     'P',
     'A',
     'φ',
     'Ulit.',
     'Wernsd.',
     'Glaeser',
     'sqq.'],
    'claim_evidence': ['G', 'P', 'A', 'φ', 'Ulit.', 'Wernsd.'],
    'reading_match': True,
    'evidence_match': False,
    'status': 'mismatch',
    'reviewer': 'SJH',
    'reviewer_notes': None,
    'review_status': 'Failed'},
   {'reading': 'om.',
    'claim_reading': None,
    'transcription_evidence': ['N', 'π', 'χ', 'p'],
    'claim_evidence': ['N', 'π', 'χ', 'p'],
    'reading_match': False,
    'evidence_match': True,
    'status': 'missing_data',
    'reviewer': 'SJH',
    'reviewer_notes': None,
    'review_status': 'Failed'},
   {'reading': 'O.',
    'claim_reading': 'O.',
    'transcription_evidence': ['ε',
     'β',
     'γ',
     'μ',
     'ρ',
     'edd. ante',

In [144]:
with open ("../converted-json/comparanda.json", "w", encoding="utf-8") as f:
    json.dump(entries, f, indent=4, ensure_ascii=False)